# Module 05: Multi-Node JAX TPU Slice Training with JobSet
In this module, you will launch a multi-node JAX TPU slice training workload on GKE using JobSet.

### Key Concepts Covered:
1. **TPU Multi-Host Slices**: Interconnecting multiple TPU hosts over high-speed Inter-Chip Interconnect (ICI).
2. **Exclusive Topology & Ports**: Specifying `alpha.jobset.sigs.k8s.io/exclusive-topology: cloud.google.com/gke-nodepool` and exposing TPU container ports `8471` (Data Link) and `8080` (Coordinator).
3. **SPMD Mesh Sharding on TPU**: Sharding large matrix operations across TPU pod slices.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import config
cfg = config.load_config("../config.env")

PROJECT_ID = cfg["PROJECT_ID"]
REGION = cfg["REGION"]
REPO = cfg["ARTIFACT_REGISTRY_REPO"]
TPU_IMAGE = cfg["TPU_IMAGE_NAME"]
TAG = cfg["IMAGE_TAG"]

TPU_FULL_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{TPU_IMAGE}:{TAG}"
print(f"Target TPU Image: {TPU_FULL_IMAGE}")

## 1. Render TPU JobSet Manifest

In [ ]:
manifest_path = Path("../manifests/jobset-tpu.yaml")
rendered_path = Path("../manifests/jobset-tpu-rendered.yaml")

with open(manifest_path, "r") as f:
    content = f.read()

content = content.replace("LOCATION-docker.pkg.dev/PROJECT_ID/ARTIFACT_REGISTRY_REPO/TPU_IMAGE_NAME:IMAGE_TAG", TPU_FULL_IMAGE)

with open(rendered_path, "w") as f:
    f.write(content)

print(f"Rendered TPU JobSet manifest saved to {rendered_path}")

## 2. Deploy JAX TPU Multi-Node JobSet

In [ ]:
!kubectl apply -f ../manifests/jobset-tpu-rendered.yaml

## 3. Monitor TPU JobSet & Pod Status

In [ ]:
import time
print("Waiting for TPU worker pods to initialize...")
for i in range(8):
    !kubectl get jobset jax-tpu-job
    !kubectl get pods -l jobset.x-k8s.io/jobset-name=jax-tpu-job -o custom-columns=POD_NAME:.metadata.name,NODE:.spec.nodeName,STATUS:.status.phase
    time.sleep(5)

## 4. Stream Logs from Multi-Node TPU Slice Workers

In [ ]:
!kubectl logs -l jobset.x-k8s.io/jobset-name=jax-tpu-job --all-containers --tail=100

## 5. Verify TPU Execution & ICI Mathematical Sum Proof
Verify from logs:
- TPU multi-host slice discovery.
- ICI interconnect `lax.psum` all-reduce output equals `3.0`.